In [1]:
import json

from models_green import GreenRide, ride_deserializer

In [3]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trips-to-postgres',
    value_deserializer=ride_deserializer
)

In [4]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [6]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_green_events
           (pickup_datetime, dropoff_datetime, pickup_date, dropoff_date, PULocationID, DOLocationID, passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
        (pickup_dt, dropoff_dt, ride.string_lpep_pickup_datetime, ride.string_lpep_dropoff_datetime,
          ride.PULocationID, ride.DOLocationID, ride.passenger_count,
         ride.trip_distance, ride.tip_amount, ride.total_amount)
    )
    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...
Inserted 1000 rows...
Inserted 2000 rows...
Inserted 3000 rows...
Inserted 4000 rows...
Inserted 5000 rows...
Inserted 6000 rows...
Inserted 7000 rows...
Inserted 8000 rows...
Inserted 9000 rows...
Inserted 10000 rows...
Inserted 11000 rows...
Inserted 12000 rows...
Inserted 13000 rows...
Inserted 14000 rows...
Inserted 15000 rows...
Inserted 16000 rows...
Inserted 17000 rows...
Inserted 18000 rows...
Inserted 19000 rows...
Inserted 20000 rows...
Inserted 21000 rows...
Inserted 22000 rows...
Inserted 23000 rows...
Inserted 24000 rows...
Inserted 25000 rows...
Inserted 26000 rows...
Inserted 27000 rows...
Inserted 28000 rows...
Inserted 29000 rows...
Inserted 30000 rows...
Inserted 31000 rows...
Inserted 32000 rows...
Inserted 33000 rows...
Inserted 34000 rows...
Inserted 35000 rows...
Inserted 36000 rows...
Inserted 37000 rows...
Inserted 38000 rows...
Inserted 39000 rows...
Inserted 40000 rows...
Inserted 41000 rows...
Inserted 420

KeyboardInterrupt: 